# Use Case Runner

One cell. Pick a `STAGE`, and it runs the matching shared pipeline script from `AMFS - ML Setup` in this same kernel, using this use case's `config.yaml`. The four pipeline scripts and `ml_pipeline_utils.py` are not copied here — only this notebook and `config.yaml` live in this use case's folder.

Runs interactively (just edit `STAGE` and run the cell) or headless — an external orchestrator (Airflow) triggers this the same way it would any other notebook run; `STAGE` just needs to be set to the right value before that run, e.g. by maintaining one small orchestrator-side override per schedule, or four thin copies of this notebook that each hardcode a different `STAGE` (still just one config.yaml and near-identical one-cell notebooks per use case).

In [ ]:
import sys, runpy
from snowflake.snowpark.context import get_active_session

# ============================================================================
# Per-use-case runner. This is the ONLY notebook a use case keeps in its own
# folder, next to config.yaml. The four pipeline scripts (training.py,
# inference_deploy.py, ground_truth_deploy.py, retraining.py) and
# ml_pipeline_utils.py live once, shared, as plain .py files in the
# "AMFS - ML Setup" folder -- this cell just runs whichever one you ask for,
# in this same Python kernel (so the already-active Snowflake session carries
# straight through -- no EXECUTE NOTEBOOK / NOTEBOOK PROJECT object involved).
# ============================================================================

# ---- 1. which stage to run -------------------------------------------------
# EDIT PER RUN (or leave as-is and re-run this cell after changing it below).
STAGE = "training"   # training | inference_deploy | ground_truth_deploy | retraining

STAGE_TO_SCRIPT = {
    "training":            "training.py",
    "inference_deploy":    "inference_deploy.py",
    "ground_truth_deploy": "ground_truth_deploy.py",
    "retraining":          "retraining.py",
}
assert STAGE in STAGE_TO_SCRIPT, f'STAGE must be one of {tuple(STAGE_TO_SCRIPT)}, got {STAGE!r}'

# ---- 2. where the shared pipeline scripts live -----------------------------
# EDIT PER ENVIRONMENT (not per use case): relative path from this use case's
# folder to the shared "AMFS - ML Setup" folder.
ML_SETUP_DIR = "../AMFS - ML Setup"

# ---- 3. this use case's own config.yaml ------------------------------------
# Relative to THIS folder (where this runner notebook and config.yaml both
# live) -- not to ML_SETUP_DIR. Left as the default; change only if this use
# case's config file is named or placed differently.
CONFIG_PATH = "config.yaml"

# ---- 4. run -----------------------------------------------------------------
sys.path.append(ML_SETUP_DIR)          # so the script's own `from ml_pipeline_utils import *` resolves
session = get_active_session()          # confirms a session is already active before handing off

script_path = f"{ML_SETUP_DIR}/{STAGE_TO_SCRIPT[STAGE]}"
print(f'STAGE={STAGE!r}  ->  running {script_path}  with CONFIG_PATH={CONFIG_PATH!r}')

runpy.run_path(script_path, init_globals={"CONFIG_PATH": CONFIG_PATH})
